# Expected value gets more expensive check

Price per unit of expected voting power must increase as level does.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# Add project source directory to import path.
import sys
import os
from pathlib import Path

parent_dir = str(os.path.join(Path.cwd().parent, 'src'))

if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

# Imports.
import math
import numpy as np
import matplotlib.pyplot as plt
import fvopn_simulations as fvopn
from scipy.interpolate import make_interp_spline

In [ ]:
rng = np.random.default_rng(42)
alpha = 2 # Root slots.
k = 2 # Multiplication factor.
size = 100 # Number of nodes.

budgets=rng.beta(.5, 8, size) # Random generated budgets.

conf = fvopn.VoteAuctionModelV2Config(
    budgets=budgets,
    root_slots=alpha,
    multiplication_factor=k,
)

model = fvopn.VoteAuctionModelV2(conf, rng)

In [ ]:
def graph_fn(fn):
    fn_vec = np.vectorize(fn)

    xs = np.arange(1, 10)
    ys = fn_vec(xs)

    return (xs, ys)

In [ ]:
from typing import Callable

def generic(fn: Callable[[float], float], x: int):
    conf = fvopn.VoteAuctionModelV2Config(
        budgets=budgets,
        root_slots=alpha,
        multiplication_factor=k,
    )

    model = fvopn.VoteAuctionModelV2(conf, rng)
    
    return fn(model.expected_value(1))

linear = lambda x: generic(lambda v: x * v, x)

logaritmic = lambda x: generic(lambda v: v*(math.log10(x)+1), x)

b = np.copy(budgets)

def protocol( x: int):  
    b[0] = x
    
    conf = fvopn.VoteAuctionModelV2Config(
        budgets=b,
        root_slots=alpha,
        multiplication_factor=k,
    )

    model = fvopn.VoteAuctionModelV2(conf, rng)
    
    res = 0

    N = 100
    
    for _ in range(N):
        a = model.run()

        a.sort_index()

        res += a.loc[0, 'votes']

    res /= N

    return res

In [ ]:
x1, y1 = graph_fn(protocol)

In [ ]:
x2, y2 = graph_fn(linear)
x3, y3 = graph_fn(logaritmic)

In [ ]:
plt.plot(x1, y1, label="Protocolo Propuesto")
plt.plot(x2, y2, label="lineal")
plt.plot(x3, y3, label="logaritmico (base 10)")

plt.title("Poder de voto")
plt.xlabel("Presupuesto")
plt.ylabel("Votos")
plt.grid()
plt.legend()
plt.show()